[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week1_data_foundations/day01_what_is_data/day01_notebook.ipynb)

---

# Day 1 / 42: What is Data in ML?
### 42 Days of AI/ML Challenge

**Author:** VaishnaviJagtap18  
**LinkedIn:** [Follow for daily posts](https://www.linkedin.com/in/VaishnaviJagtap18)  
**GitHub Repo:** [42-days-aiml-challenge](https://github.com/VaishnaviJagtap18/42-days-aiml-challenge)

---

## What You Will Learn Today

- The 3 types of data every ML engineer works with
- Why identifying data type is the first step in every ML project
- How to load and inspect each data type in Python
- What happens in production when you treat all data types the same way
- How to classify any dataset you encounter on the job

**Estimated time:** 45-60 minutes  
**Prerequisites:** Python basics, pandas installed  

---

## Setup

Run this cell first. It installs all dependencies and confirms your environment is ready.

In [ ]:
# Install dependencies
# All of these are pre-installed in Google Colab.
# If running locally and something is missing, uncomment and run:
# !pip install pandas numpy matplotlib pillow

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
import os
from PIL import Image
import io
import warnings
warnings.filterwarnings('ignore')

# Color palette for this series
CYAN   = '#00C4CC'
ORANGE = '#F4A233'
PURPLE = '#7C4DFF'
NAVY   = '#0A1628'
GREEN  = '#00C853'

print('Environment ready.')
print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'Python  : 3.x required - you are good to go')

---

## Part 1: The 3 Types of Data

Every ML project starts with data. Before you write a single line of preprocessing code, you need to know what type of data you are working with. The type determines:

- Which pipeline steps you need
- Which algorithms you can use directly vs which need a conversion step first
- How long preprocessing will take
- What can break in production

There are 3 types:

| Type | Definition | Schema | Example |
|------|-----------|--------|--------|
| **Structured** | Rows and columns. Every field has a label. | Fixed | Transactions, sensor readings, stock prices |
| **Semi-Structured** | Has some organization but no strict schema. | Flexible | JSON logs, XML files, emails |
| **Unstructured** | No fixed format. Needs a conversion step before a model can use it. | None | Images, audio, raw text, video |

> **Production reality:** Most real datasets are a mix of all 3 types. Spotify's recommendation system handles structured play history, semi-structured playlist metadata, and unstructured podcast audio simultaneously using 3 separate pipelines before any model sees the data.

---
### 1.1 Structured Data

Structured data lives in rows and columns. Every field has a name and a type. It loads directly into a pandas DataFrame and is ready for most ML algorithms with minimal preprocessing.

We will simulate a Spotify-style user play history dataset.

In [ ]:
# STRUCTURED DATA: Spotify user play history
# This is the kind of data that feeds directly into a recommendation model.

np.random.seed(42)
n = 20

play_history = pd.DataFrame({
    'user_id':      np.random.randint(1001, 1020, n),
    'song_id':      np.random.randint(5001, 5050, n),
    'play_count':   np.random.randint(1, 100, n),
    'skipped':      np.random.choice([True, False], n, p=[0.3, 0.7]),
    'rating':       np.random.choice([1, 2, 3, 4, 5, None], n),
    'last_played':  pd.date_range('2024-01-01', periods=n, freq='D')
})

print('=== STRUCTURED DATA: Spotify Play History ===')
print(f'Shape: {play_history.shape}')
print(f'\nFirst 5 rows:')
print(play_history.head())
print(f'\nData types:')
print(play_history.dtypes)
print(f'\nMissing values:')
print(play_history.isnull().sum())
print(f'\nBasic statistics:')
print(play_history.describe())

In [ ]:
# Key observations about structured data:
# 1. Every column has a clear dtype
# 2. You can call .describe() directly
# 3. Missing values are easy to detect with .isnull()
# 4. Ready for pandas operations immediately

print('=== What you can do IMMEDIATELY with structured data ===')
print(f'Average play count: {play_history["play_count"].mean():.1f}')
print(f'Skip rate: {play_history["skipped"].mean()*100:.1f}%')
print(f'Most active user: {play_history.groupby("user_id")["play_count"].sum().idxmax()}')
print(f'\nCorrelation between play_count and skipped:')
print(f'{play_history[["play_count","skipped"]].corr().iloc[0,1]:.3f}')
print('\nStructured data requires NO conversion step before analysis.')

---
### 1.2 Semi-Structured Data

Semi-structured data has some organization but no strict schema. JSON is the most common format in ML pipelines. The challenge: two records can have completely different fields, and you need to extract and flatten what you need before modelling.

In [ ]:
# SEMI-STRUCTURED DATA: Spotify playlist metadata
# Notice how each playlist JSON can have different fields.
# This is what makes it semi-structured, not fully structured.

playlists = [
    {
        'playlist_id': 'PL001',
        'name': 'Morning Vibes',
        'created_by': 'user_1001',
        'collaborative': False,
        'tracks': [
            {'track_id': 5001, 'title': 'Song A', 'duration_ms': 210000},
            {'track_id': 5002, 'title': 'Song B', 'duration_ms': 195000},
            {'track_id': 5003, 'title': 'Song C', 'duration_ms': 230000}
        ]
        # No 'description' field here
    },
    {
        'playlist_id': 'PL002',
        'name': 'Workout Mix',
        'created_by': 'user_1005',
        'collaborative': True,
        'collaborators': ['user_1008', 'user_1012'],  # Extra field not in PL001
        'description': 'High energy tracks for gym sessions',  # Extra field
        'tracks': [
            {'track_id': 5010, 'title': 'Song D', 'duration_ms': 180000, 'bpm': 140},  # 'bpm' not in PL001 tracks
            {'track_id': 5011, 'title': 'Song E', 'duration_ms': 200000, 'bpm': 135}
        ]
    },
    {
        'playlist_id': 'PL003',
        'name': 'Study Focus',
        'created_by': 'user_1003',
        'collaborative': False,
        'mood_tags': ['calm', 'focus', 'instrumental'],  # Another unique field
        'tracks': [
            {'track_id': 5020, 'title': 'Song F', 'duration_ms': 320000}
        ]
    }
]

print('=== SEMI-STRUCTURED DATA: Playlist Metadata (JSON) ===')
print(f'Number of playlists: {len(playlists)}')
print(f'\nPL001 keys: {list(playlists[0].keys())}')
print(f'PL002 keys: {list(playlists[1].keys())}')
print(f'PL003 keys: {list(playlists[2].keys())}')
print(f'\nNotice: Each playlist has DIFFERENT fields.')
print(f'This inconsistency is what makes it semi-structured.')

In [ ]:
# To use semi-structured data in ML, you must FLATTEN it first.
# Extract the fields you need into a structured DataFrame.

def flatten_playlist(pl):
    """Extract structured features from semi-structured playlist JSON."""
    return {
        'playlist_id':    pl['playlist_id'],
        'name':           pl['name'],
        'track_count':    len(pl['tracks']),
        'collaborative':  int(pl['collaborative']),
        'has_description': int('description' in pl),
        'has_mood_tags':  int('mood_tags' in pl),
        'avg_duration_ms': np.mean([t['duration_ms'] for t in pl['tracks']]),
        'has_collaborators': int('collaborators' in pl)
    }

playlists_df = pd.DataFrame([flatten_playlist(pl) for pl in playlists])

print('=== After flattening: semi-structured -> structured ===')
print(playlists_df)
print(f'\nShape: {playlists_df.shape}')
print(f'dtypes:\n{playlists_df.dtypes}')
print('\nNow this is ready to feed into an ML model.')

---
### 1.3 Unstructured Data

Unstructured data has no fixed format at all. A model cannot consume a raw image, audio file, or text string directly. You need a feature extraction step first that converts it into numbers the model understands.

- Image -> pixel array -> feature vector or embedding
- Audio -> spectrogram -> feature vector
- Text -> tokens -> embeddings

In [ ]:
# UNSTRUCTURED DATA: Simulating an album cover image
# In Spotify's pipeline, album cover images are processed
# through a CNN to extract visual style features.

# Create a synthetic album cover (3 color blocks representing different moods)
np.random.seed(42)

# Simulate 3 different album covers with different dominant colors
def create_album_cover(dominant_color, noise=30, size=(100, 100)):
    """Create a synthetic album cover image with a dominant color."""
    img_array = np.full((*size, 3), dominant_color, dtype=np.uint8)
    img_array = img_array + np.random.randint(-noise, noise, (*size, 3))
    img_array = np.clip(img_array, 0, 255).astype(np.uint8)
    return Image.fromarray(img_array)

# 3 album covers with different moods (warm, cool, dark)
covers = {
    'Morning Vibes (warm)':  create_album_cover([220, 150, 80]),
    'Workout Mix (energetic)': create_album_cover([80, 200, 180]),
    'Study Focus (dark)':   create_album_cover([50, 50, 100])
}

print('=== UNSTRUCTURED DATA: Album Cover Images ===')
for name, img in covers.items():
    print(f'\n{name}')
    print(f'  Type: {type(img)}')
    print(f'  Size (width x height): {img.size}')
    print(f'  Mode (color channels): {img.mode}')
    print(f'  As numpy array shape: {np.array(img).shape}')

print('\nRaw image cannot be fed to a model directly.')
print('It must be converted to a numerical array first.')

In [ ]:
# Visualize the 3 album covers
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('Unstructured Data: Album Cover Images', fontsize=14, fontweight='bold')

for ax, (name, img) in zip(axes, covers.items()):
    ax.imshow(img)
    ax.set_title(name, fontsize=9, pad=8)
    ax.axis('off')

plt.tight_layout()
plt.show()
print('These look like images to us. To a model they are meaningless until converted to numbers.')

In [ ]:
# FEATURE EXTRACTION: Converting unstructured image to structured features
# This is the step that makes unstructured data usable for ML.

def extract_image_features(img, name):
    """
    Extract structured features from an unstructured image.
    In production, a CNN would do this automatically.
    Here we manually extract simple color statistics as a demo.
    """
    arr = np.array(img).astype(float)
    r, g, b = arr[:,:,0], arr[:,:,1], arr[:,:,2]

    return {
        'name':          name,
        'mean_red':      round(r.mean(), 2),
        'mean_green':    round(g.mean(), 2),
        'mean_blue':     round(b.mean(), 2),
        'brightness':    round(arr.mean(), 2),
        'red_dominant':  int(r.mean() > g.mean() and r.mean() > b.mean()),
        'dark_image':    int(arr.mean() < 100),
        'pixel_count':   arr.shape[0] * arr.shape[1]
    }

features_df = pd.DataFrame([extract_image_features(img, name) for name, img in covers.items()])

print('=== After feature extraction: unstructured -> structured ===')
print(features_df.to_string(index=False))
print(f'\nShape: {features_df.shape}')
print('\nNow this structured feature table can feed into any ML model.')
print('This is exactly what CNNs do automatically at scale in production.')

---

## Part 2: Side-by-Side Comparison

Now that you have seen all 3 types in action, let's put them side by side so the differences are clear.

In [ ]:
# Side-by-side comparison table

comparison = pd.DataFrame({
    'Property': [
        'Schema',
        'Storage Format',
        'ML Pipeline Step',
        'pandas .read() works directly?',
        'Common in production?',
        'Example Tool',
        'Data Volume in Real Systems'
    ],
    'Structured': [
        'Fixed',
        'CSV, SQL, Parquet',
        'Load -> Preprocess -> Model',
        'Yes',
        'Yes (40%)',
        'pandas, PySpark',
        '~20% of total data'
    ],
    'Semi-Structured': [
        'Flexible',
        'JSON, XML, YAML',
        'Parse -> Flatten -> Preprocess -> Model',
        'Partially',
        'Yes (30%)',
        'json, xmltodict',
        '~15% of total data'
    ],
    'Unstructured': [
        'None',
        'jpg, mp3, txt, mp4',
        'Extract Features -> Preprocess -> Model',
        'No',
        'Yes (30%)',
        'PIL, librosa, spaCy',
        '~80% of total data'
    ]
})

print('=== Complete Comparison ===')
print(comparison.to_string(index=False))

In [ ]:
# Visual: Pipeline complexity per data type

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('ML Pipeline Steps Per Data Type', fontsize=14, fontweight='bold', y=1.02)

pipelines = {
    'Structured\n(e.g. CSV)': {
        'steps': ['Load Data', 'Handle Missing', 'Encode/Scale', 'Train Model'],
        'color': CYAN
    },
    'Semi-Structured\n(e.g. JSON)': {
        'steps': ['Load JSON', 'Parse Fields', 'Flatten', 'Handle Missing', 'Encode/Scale', 'Train Model'],
        'color': ORANGE
    },
    'Unstructured\n(e.g. Image)': {
        'steps': ['Load File', 'Decode', 'Resize/Normalize', 'Extract Features', 'Flatten to Vector', 'Handle Missing', 'Scale', 'Train Model'],
        'color': PURPLE
    }
}

for ax, (title, info) in zip(axes, pipelines.items()):
    steps = info['steps']
    color = info['color']
    n_steps = len(steps)

    ax.set_xlim(0, 1)
    ax.set_ylim(-0.5, n_steps - 0.5)
    ax.axis('off')
    ax.set_title(title, fontsize=11, fontweight='bold', pad=10)

    for i, step in enumerate(reversed(steps)):
        y = i
        rect = mpatches.FancyBboxPatch(
            (0.05, y - 0.35), 0.9, 0.7,
            boxstyle='round,pad=0.05',
            facecolor=color,
            edgecolor='white',
            linewidth=1.5,
            alpha=0.85
        )
        ax.add_patch(rect)
        ax.text(0.5, y, step, ha='center', va='center',
                fontsize=9, fontweight='bold', color='white')

        if i < n_steps - 1:
            ax.annotate('', xy=(0.5, i + 0.38), xytext=(0.5, i + 0.62),
                       arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

    ax.text(0.5, -0.45, f'{n_steps} steps', ha='center', fontsize=9,
            color='gray', style='italic')

plt.tight_layout()
plt.show()
print('Unstructured data requires the most pipeline steps.')
print('This is why data type identification is Step 1 of every ML project.')

---

## Part 3: Classification Exercise

Before looking at the answers, classify each dataset yourself.

For each one, decide: **Structured, Semi-Structured, or Unstructured?**

Think about:
- Does it have a fixed schema?
- Can you load it directly into a pandas DataFrame without any parsing?
- Does it need a feature extraction step before a model can use it?

In [ ]:
# EXERCISE: Classify these 10 datasets
# Read each one and write your answer before running the next cell.

datasets_to_classify = [
    {'id': 1,  'dataset': 'Customer transactions CSV with columns: user_id, amount, timestamp, merchant'},
    {'id': 2,  'dataset': 'Folder of 50,000 product photos (.jpg files)'},
    {'id': 3,  'dataset': 'User activity logs in JSON where each event has different fields'},
    {'id': 4,  'dataset': 'Employee salary spreadsheet with fixed columns'},
    {'id': 5,  'dataset': 'Customer support audio call recordings (.wav files)'},
    {'id': 6,  'dataset': 'Email inbox exported as .eml files (headers + body mixed)'},
    {'id': 7,  'dataset': 'Stock price time series: date, open, high, low, close, volume'},
    {'id': 8,  'dataset': 'Raw social media posts scraped as plain text (.txt files)'},
    {'id': 9,  'dataset': 'XML configuration files where optional tags vary per record'},
    {'id': 10, 'dataset': 'Hospital patient records with fixed columns: age, diagnosis, lab_results'},
]

print('=== CLASSIFY THESE 10 DATASETS ===')
print('Write your answer for each before running the solution cell.\n')
for d in datasets_to_classify:
    print(f"Dataset {d['id']:2d}: {d['dataset']}")
    print(f"          Your answer: _______________\n")

In [ ]:
# SOLUTION: Run this only after you have written your own answers above.

solutions = [
    {'id': 1,  'answer': 'Structured',      'reason': 'Fixed columns, loads directly into pandas'},
    {'id': 2,  'answer': 'Unstructured',    'reason': 'Raw image files, need feature extraction (CNN) first'},
    {'id': 3,  'answer': 'Semi-Structured', 'reason': 'JSON with inconsistent fields per event'},
    {'id': 4,  'answer': 'Structured',      'reason': 'Fixed schema spreadsheet, direct pandas load'},
    {'id': 5,  'answer': 'Unstructured',    'reason': 'Audio files, need spectrogram or embedding extraction first'},
    {'id': 6,  'answer': 'Semi-Structured', 'reason': 'Emails have loose structure (headers + body) but vary per email'},
    {'id': 7,  'answer': 'Structured',      'reason': 'Fixed OHLCV schema, standard time series format'},
    {'id': 8,  'answer': 'Unstructured',    'reason': 'Free-form text, needs tokenization and embeddings first'},
    {'id': 9,  'answer': 'Semi-Structured', 'reason': 'XML has hierarchy but optional tags make it inconsistent'},
    {'id': 10, 'answer': 'Structured',      'reason': 'Fixed columns even if some values are complex'},
]

print('=== SOLUTIONS ===')
print(f'{"ID":<4} {"Answer":<20} {"Reason"}')
print('-' * 75)
for s in solutions:
    print(f"{s['id']:<4} {s['answer']:<20} {s['reason']}")

print('\n=== SCORING ===')
print('10/10 : Ready for Day 2. You understand data types.')
print('7-9   : Good. Review the ones you missed above.')
print('< 7   : Re-read Part 1 and the comparison table, then try again.')

---

## Part 4: The Production Problem

This is the most important section. Knowing data types conceptually is not enough. Understanding what breaks in production when you ignore them is what makes you a professional ML engineer.

In [ ]:
# THE PRODUCTION PROBLEM:
# A fintech MLE builds a credit scoring model with 3 data sources.
# They treat all 3 the same way. The model breaks in production.

print('=== PRODUCTION SCENARIO: Credit Scoring Model ===')
print()

data_sources = pd.DataFrame({
    'Source': [
        'Transaction history',
        'Customer support chat logs',
        'KYC JSON documents'
    ],
    'Type': [
        'Structured',
        'Unstructured',
        'Semi-Structured'
    ],
    'Update Frequency': [
        'Daily',
        'Real-time',
        'Once a year'
    ],
    'Schema Changes?': [
        'Rarely',
        'Every 2 weeks',
        'Never'
    ]
})

print(data_sources.to_string(index=False))

print()
print('WHAT THE MLE DID (WRONG):')
print('  Built one unified pipeline treating all 3 sources the same.')
print('  Model trained fine. Shipped to production.')
print()
print('WHAT HAPPENED:')
print('  Pipeline broke every 2 weeks when chat log schema changed.')
print('  Model went down. Credits could not be scored. Revenue lost.')
print()
print('ROOT CAUSE:')
print('  Unstructured chat logs need a feature extraction step.')
print('  When the chat platform changed its JSON structure, the extraction broke.')
print('  One pipeline for 3 data types = one point of failure for all 3.')

In [ ]:
# THE FIX: Separate pipelines per data type
# This is the production-grade architecture.

print('=== THE FIX: Separate Ingestion Pipelines ===')
print()

architecture = pd.DataFrame({
    'Pipeline': [
        'Pipeline 1: Structured',
        'Pipeline 2: Semi-Structured',
        'Pipeline 3: Unstructured'
    ],
    'Input': [
        'Transaction CSV',
        'KYC JSON',
        'Chat logs'
    ],
    'Key Step': [
        'Schema validation -> Scale',
        'Parse + Flatten -> Validate -> Scale',
        'Schema validation -> Extract embeddings -> Scale'
    ],
    'Output': [
        'Feature vector A',
        'Feature vector B',
        'Feature vector C'
    ],
    'Isolated Failure?': [
        'Yes',
        'Yes',
        'Yes'
    ]
})

print(architecture.to_string(index=False))
print()
print('FINAL STEP: Concatenate feature vectors A + B + C -> Train model')
print()
print('RESULT:')
print('  Chat log schema changes -> Only Pipeline 3 affected')
print('  Model continues running on Pipelines 1 and 2')
print('  Engineers fix Pipeline 3 independently')
print()
print('The model did not change. The architecture did.')
print('This is production ML engineering.')

In [ ]:
# Simulate the full pipeline joining all 3 data types into one feature matrix

np.random.seed(42)
n_users = 100

# Pipeline 1 output: structured transaction features
transaction_features = pd.DataFrame({
    'user_id':           range(n_users),
    'avg_transaction':   np.random.normal(500, 200, n_users).round(2),
    'transaction_count': np.random.randint(1, 50, n_users),
    'days_since_last':   np.random.randint(1, 90, n_users)
})

# Pipeline 2 output: semi-structured KYC features (after flattening)
kyc_features = pd.DataFrame({
    'user_id':           range(n_users),
    'has_pan':           np.random.choice([0, 1], n_users, p=[0.1, 0.9]),
    'has_aadhar':        np.random.choice([0, 1], n_users, p=[0.05, 0.95]),
    'kyc_score':         np.random.randint(60, 100, n_users)
})

# Pipeline 3 output: unstructured chat features (after embedding extraction)
chat_features = pd.DataFrame({
    'user_id':           range(n_users),
    'sentiment_score':   np.random.uniform(-1, 1, n_users).round(3),
    'complaint_count':   np.random.randint(0, 10, n_users),
    'chat_embedding_dim1': np.random.normal(0, 1, n_users).round(4),
    'chat_embedding_dim2': np.random.normal(0, 1, n_users).round(4)
})

# Join all 3 pipelines on user_id
final_features = transaction_features \
    .merge(kyc_features, on='user_id') \
    .merge(chat_features, on='user_id')

print('=== Final Feature Matrix (3 pipelines joined) ===')
print(f'Shape: {final_features.shape}')
print(f'Columns: {list(final_features.columns)}')
print(f'\nFirst 5 rows:')
print(final_features.head())
print(f'\nThis is what goes into the credit scoring model.')
print(f'3 separate pipelines. 1 clean feature matrix.')

---

## Part 5: Reflection Prompt

Before you move to Day 2, answer these 3 questions in the cell below. Write your actual answers, not ideal textbook answers. These force you to connect today's concept to real work.

1. What type of data does your current project or the job you want to apply for deal with most?
2. If you had to build an ML pipeline today, which data type would you find hardest to handle and why?
3. Think of one company whose ML product you use daily. What mix of data types do you think their model consumes?

In [ ]:
# Write your answers here as strings.
# There are no wrong answers. The point is to think.

my_answers = {
    'Q1_my_data_type':       'YOUR ANSWER HERE',
    'Q2_hardest_to_handle':  'YOUR ANSWER HERE',
    'Q3_company_data_mix':   'YOUR ANSWER HERE'
}

print('=== Your Reflection ===')
for q, a in my_answers.items():
    print(f'{q}: {a}')
print()
print('Drop your answers in the LinkedIn comments on today\'s post.')
print('Reading what others write is part of the learning.')

---

## Summary: What You Covered Today

| # | Concept | Status |
|---|---------|--------|
| 1 | 3 types of data: Structured, Semi-Structured, Unstructured | Done |
| 2 | Loaded and inspected all 3 types in Python | Done |
| 3 | Flattened semi-structured JSON into a DataFrame | Done |
| 4 | Extracted features from unstructured image data | Done |
| 5 | Saw how Spotify handles all 3 types in one system | Done |
| 6 | Understood the production failure pattern and its fix | Done |
| 7 | Classified 10 real-world datasets correctly | Done |
| 8 | Built a 3-pipeline feature matrix from scratch | Done |

---

## What is Next: Day 2

**Data Types and Distributions**

You will learn:
- Numerical vs categorical vs ordinal vs boolean data types
- Why assigning the wrong dtype silently breaks downstream models
- How to identify and fix type errors in a real dataset
- What skewness looks like and why it matters before modelling

---

## Resources

- Airbnb Engineering Blog on data infrastructure: medium.com/airbnb-engineering
- Spotify Engineering Blog: engineering.atspotify.com
- Snowflake unstructured data docs: docs.snowflake.com
- Great Expectations for schema validation: github.com/great-expectations/great_expectations

---

**Follow the series:**  
LinkedIn: VaishnaviJagtap18  
GitHub: https://github.com/VaishnaviJagtap18/42-Days-0f-ML-Challenge  
#42DaysOfML